# 01 — Parameter and Hyperparameter Selection

Global tuning across the six datasets. A deterministic 10% holdout is taken from each training split; public dev is reserved for the final labeled comparison. Bayesian optimization runs the same MIGRES loop for each trial and averages final raw retrieval Recall@K across datasets. The budget is explicit: 3 K values × (3 initial + 7 BO) = 30 trials; 8 tuning questions per dataset per trial.

In [ ]:
from pathlib import Path
import sys, json, yaml, platform
ROOT = Path.cwd()
if not (ROOT / "pyproject.toml").exists():
    ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT / "src"))
print("Project root:", ROOT)
try:
    import torch
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    print("Torch:", torch.__version__)
    print("CUDA available:", torch.cuda.is_available())
    print("Selected device:", DEVICE)
    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))
except Exception as e:
    DEVICE = "cpu"
    print("Torch/GPU probe unavailable:", type(e).__name__, e)
PLAN = yaml.safe_load((ROOT / "configs" / "experiments.yaml").read_text())
BASE_CONFIG = [ROOT / "configs" / "default.yaml"]


In [ ]:
RUN_EXPERIMENT = False
VALIDATION_SUBSET_PER_DATASET = PLAN["parameter_selection"]["validation_subset_per_dataset"]
SELECTED_PATH = ROOT / "outputs/experiments/01_parameter_selection/selected_config.json"
RUNS_ROOT = ROOT / "outputs/experiments/01_parameter_selection/bayes_runs"


In [ ]:
from sar_migres.config import load_config
from sar_migres.experiment_runner import load_all_dataset_items_for_phase, validation_mean_recall_objective, run_pipeline_experiment, retrieval_metrics_from_run
from sar_migres.optimization import optimize_weights
from sar_migres.scoring import SARWeights

cfg = load_config(*BASE_CONFIG)
all_tuning = {}
if RUN_EXPERIMENT:
    all_tuning = load_all_dataset_items_for_phase(cfg, "tuning", VALIDATION_SUBSET_PER_DATASET)
    print({k: len(v) for k,v in all_tuning.items()})
print('Scope:', PLAN['parameter_selection']['bayesian_optimization']['scope'])
print('Budget:', PLAN['parameter_selection']['bayesian_optimization']['budget_note'])

if RUN_EXPERIMENT:
    if not all_tuning:
        raise RuntimeError('No tuning data loaded; verify data/raw paths in configs/default.yaml')
    candidates=[]
    for k in PLAN['parameter_selection']['k_grid']:
        counter={'n':0}
        def objective(w):
            counter['n'] += 1
            vals=[]
            for ds, items in all_tuning.items():
                ov={"retrieval":{"method":"sar"},"sar":{"alpha":w.alpha,"beta":w.beta,"gamma":w.gamma,"k":int(k),"weighting":"static"},"data":{"dataset":ds,"dataset_path":cfg.data.dataset_paths[ds],"split":"train"},"run":{"name":f'k{k}_trial{counter["n"]:03d}_{ds}',"output_dir":str(RUNS_ROOT),"resume":False}}
                pipe,_=run_pipeline_experiment(BASE_CONFIG,items,overrides=ov,experiment_name=f'k{k}_trial{counter["n"]:03d}_{ds}')
                m=retrieval_metrics_from_run(pipe.run_dir,items,pipe.corpus)
                vals.append(m['final_mean']['recall_at_k'])
            vals=[v for v in vals if v==v]
            return float(sum(vals)/len(vals)) if vals else 0.0
        best, score, history = optimize_weights(objective, init_points=3, n_iter=7, seed=13)
        candidates.append({"k":int(k),"weights":best.as_tuple(),"objective":score,"history":history})
    selected=max(candidates,key=lambda r:r['objective'])
    result={"objective":"mean_final_recall_at_k","scope":"global_across_six_datasets","split":"10% train holdout","selected_k":selected['k'],"selected_weights":selected['weights'],"candidates":candidates}
    SELECTED_PATH.parent.mkdir(parents=True,exist_ok=True)
    SELECTED_PATH.write_text(json.dumps(result,indent=2),encoding='utf-8')
    print(json.dumps({k:result[k] for k in ('objective','scope','selected_k','selected_weights')},indent=2))
else:
    print('Preflight only. Tuning questions per dataset:', VALIDATION_SUBSET_PER_DATASET)
